In [1]:
import numpy as np
from aeon.datasets import load_classification
from sklearn.model_selection import train_test_split
import pandas as pd
from pymdma.time_series.measures.synthesis_val import DTW, SpectralCoherence
from source.morph2 import Morph
from sklearn.preprocessing import LabelEncoder
import time
import warnings
warnings.filterwarnings('ignore')

In [ ]:
ECG_datasets = ['ECG200', 'Epilepsy', 'ECGFiveDays', 'StandWalkJump', 
        'TwoLeadECG','NerveDamage' , 'MedicalImages',
        'Colposcopy', 'EyesOpenShut', 'ToeSegmentation1']
    

results = []

for df_name in ECG_datasets:
    try:
        # Load Dataset ===================================
        X, y = load_classification(df_name)
        le = LabelEncoder()
        y = le.fit_transform(y)
    except Exception as e:
        print(f'{df_name}: Dataset Not Available - {str(e)}')
        continue

    print("-" * 70)
    print("Dataset Name:", df_name)
    print("-" * 50)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Loop Through each Class ===================================
    for c in np.unique(y):
        start_class = time.time()
        print(f'Processing Class: {c}')

        # Compute Morphing ===================================
        linear_m = Morph(X_test, y_test, c, use_dba=False)
        linear_m.get_DTWGlobalBorderline(X_test.shape[0])

        pathi_m = Morph(X_test, y_test, c, use_dba=True)
        pathi_m.get_DTWGlobalBorderline(X_test.shape[0])

        dba = pathi_m.get_AllMorphs(granularity=10, keep_original_serie=False)
        linear = linear_m.get_AllMorphs(granularity=10, keep_original_serie=False)

        dtw = DTW()
        spectral = SpectralCoherence()

        DTW_dba = dtw.compute(X_test, dba).dataset_level.value
        Spectral_dba = spectral.compute(X_test, dba).dataset_level.value

        DTW_linear = dtw.compute(X_test, linear).dataset_level.value
        Spectral_linear = spectral.compute(X_test, linear).dataset_level.value

        DTW_standard = dtw.compute(X_test, X_test).dataset_level.value
        Spectral_standard = spectral.compute(X_test, X_test).dataset_level.value


        line = [df_name, c, 
                np.round(DTW_dba,4), np.round(Spectral_dba,4), 
                np.round(DTW_linear,4), np.round(Spectral_linear,4), 
                np.round(DTW_standard,4), np.round(Spectral_standard,4)]
    
        results.append(line)


    columns = ['datset', 'class', 
                'DTW_DBA', 'Spectral_DBA', 
                'DTW_Linear', 'Spectral_Linear', 
                'DTW_Standard', 'Spectral_Standard']
    results_df = pd.DataFrame(results, columns=columns)
    results_df.to_csv('final/synthEval.csv')